In [10]:
!pip install groq python-dotenv -q
from google.colab import drive
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

drive.mount('/content/drive')
env_path = '/content/drive/MyDrive/Agentic RAG Team 1/.env'
load_dotenv(dotenv_path=env_path)


%run "/content/drive/MyDrive/Agentic RAG Team 1/Dataset.ipynb"
%run "/content/drive/MyDrive/Agentic RAG Team 1/ReAct loop.ipynb"
%run "/content/drive/MyDrive/Agentic RAG Team 1/Prompt_and_llm_integration.ipynb"
%run "/content/drive/MyDrive/Agentic RAG Team 1/Retriever.ipynb"
%run "/content/drive/MyDrive/Agentic RAG Team 1/Metrics_and_Logger.ipynb"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
items=[TitleSentenceItem(title='Will Kemp (actor, born 1977)', sentences=['William "Will" Kemp (born 29 June 1977) is an English actor and dancer.']), TitleSentenceItem(title='Slumber (film)', sentences=['Slumber is an upcoming American-British supernatural horror-thriller film directed by Jonathan Hopkins and co-written by Richard Hobley and Hopkins.', ' The film stars Maggie Q, Will Kemp, Sylvester McCoy and William Hope.', ' Principal photography began on February 11, 2016 in UK.']), TitleSentenceItem(title='Allegra Fuller Snyder', sentences=['Allegra Fuller Snyder is a dance ethnologist (ethnochoreologist), choreographer, professor and author specializing on dance and culture.', ' Her research focuses on dances among Native Amer

FlattenedContextMap(items=[FlattenedContextItem(title='Will Kemp (actor, born 1977)', sentence_string='William "Will" Kemp (born 29 June 1977) is an English actor and dancer.'), FlattenedContextItem(title='Slumber (film)', sentence_string='Slumber is an upcoming American-British supernatural horror-thriller film directed by Jonathan Hopkins and co-written by Richard Hobley and Hopkins.  The film stars Maggie Q, Will Kemp, Sylvester McCoy and William Hope.  Principal photography began on February 11, 2016 in UK.'), FlattenedContextItem(title='Allegra Fuller Snyder', sentence_string='Allegra Fuller Snyder is a dance ethnologist (ethnochoreologist), choreographer, professor and author specializing on dance and culture.  Her research focuses on dances among Native American tribes particularly the Yaqui, and on dance among several ethnic groups in Africa and Asia.  She is Professor Emerita of dance ethnology from the University of California at Los Angeles (UCLA).'), FlattenedContextItem(ti

{
  "question": "American theatre choreographer and director, Sam Pinkleton, choreographed a 2017 production, based on a 2001 romantic comedy film with a book by who?",
  "context": [
    {
      "title": "Amélie (musical)",
      "sentence": "Amélie is a musical based on the 2001 romantic comedy film with music by Daniel Messé, lyrics by Messé and Nathan Tysen and a book by Craig Lucas.  The musical premiered at Berkeley Repertory Theatre in September 2015.  The musical opened on Broadway in March 2017 and closed on May 21, 2017."
    },
    {
      "title": "Stephen Mear",
      "sentence": "Stephen Mear (born 1964) is an English dancer and choreographer best known for his award-winning work in musical theatre.  In 2005, Mear and co-choreographer Matthew Bourne won the Laurence Olivier Award for \"Best Choreography\", for their work on the new West End musical \"Mary Poppins\".  This production later transferred to Broadway in 2006, being nominated for the Tony Award for \"Best Chore

**Main Loop**

Parsing the input csv file

In [11]:
csv_path = '/content/drive/MyDrive/Agentic RAG Team 1/secret_test_set.csv'
json_path = '/content/drive/MyDrive/Agentic RAG Team 1/test_dataset.json'

df_test = pd.read_csv(csv_path)

processed_list = normalise_all_rows(df_test)
valid_rows = [row for row in processed_list if row is not None]

print(f"\nSuccessfully normalized {len(valid_rows)} out of {len(df_test)} rows.")

if len(valid_rows) > 0:
    create_json(valid_rows,json_path)
    print("JSON file successfully generated via create_json!")
else:
    print("No valid rows to save. Check the error logs printed by normalise_all_rows.")


Successfully normalized 15 out of 15 rows.
JSON file successfully generated via create_json!


Getting parsed data from json file and running the agent loop

In [12]:
with open(json_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} questions from JSON. Starting Agent Loop...\n")

title_list, sentence_list, question_list = extract_data(dataset)
document = make_document(title_list, sentence_list)

global tf_idf_vectorizer, vectorized_matrix

tf_idf_vectorizer = TfidfVectorizer()
vectorized_matrix = tf_idf_vectorizer.fit_transform(document)

generated_answers = []
collected_run_records = []

for index, row in enumerate(dataset):
    question_text = row['question']

    print(f"\nProcessing Q{index + 1}: {question_text} ")

    agent_answer, trajectory = run_agent(
        question=question_text,
        context=document
    )

    if agent_answer is None:
        agent_answer = "Budget exhausted or invalid action."

    generated_answers.append(agent_answer)

    collected_run_records.append({
        'index': index,
        'question': question_text,
        'agent_answer': agent_answer,
        'gold_answer': row.get('answer', ''),
        'trajectory': trajectory,
        'supporting_facts': row.get('supporting_facts', [])
    })

    print(f"Agent's Answer: {agent_answer}")

    time.sleep(12)

print("\n All trajectories are saved in memory.")

Loaded 15 questions from JSON. Starting Agent Loop...


Processing Q1: Wendy Lee Gramm is the wife of the congressman who represented which state? 
Agent's Answer: Texas

Processing Q2: What form of play does Merle M. Rasmussen and Top Secret have in common? 
Agent's Answer: espionage role‑playing game

Processing Q3: Jorge Paulo Agostinho Mendes, better known simply as Jorge Mendes, is a Portuguese football agent, Mendes is among the most influential football agents in the world, with clients including David de Gea, a Spanish professional footballer who plays as a goalkeeper for which English club, and the Spain national team? 
Agent's Answer: Manchester United

Processing Q4: On what peninsula is the most important work of Francis William Deas ? 
Agent's Answer: Cowal peninsula

Processing Q5: Provincial road N701 connects the Rijiksweg 6 to what city, named after Cornelis Lely? 
Agent's Answer: Lelystad

Processing Q6: Sylvia's Lovers was written by the novelist of what nationality?

Logging the agent's trajectory and evaluating its response

In [13]:
print("\n--- Evaluation & Logging ---")

df_test = df_test.drop(columns=['generated_ans'], errors='ignore')
df_test['gen_ans'] = generated_answers

allowed_columns = ['question', 'context', 'answer', 'supporting_facts', 'gen_ans']
final_columns = [col for col in allowed_columns if col in df_test.columns]
df_test = df_test[final_columns]

df_test.to_csv(csv_path, index=False)
print(f"\nFinal answers saved directly into {csv_path}")

if 'answer' in df_test.columns and 'supporting_facts' in df_test.columns:

    log_file_path = '/content/drive/MyDrive/Agentic RAG Team 1/agent_trajectories.txt'
    init_logger(log_file_path)

    for record in collected_run_records:
        log_eval_item(
            record['index'],
            record['question'],
            record['trajectory'],
            record['gold_answer'],
            log_file_path
        )
    print(f"Trajectories logged to {log_file_path}")

    eval_records = get_eval_records(collected_run_records)

    final_metrics = calculate_final_metrics(eval_records)

    print("\nFINAL SYSTEM METRICS:")
    for metric_name, value in final_metrics.items():
        print(f" -> {metric_name}: {value:.4f}")

    metrics_df = pd.DataFrame([final_metrics])
    metrics_csv_path = '/content/drive/MyDrive/Agentic RAG Team 1/evaluation_metrics.csv'
    metrics_df.to_csv(metrics_csv_path, index=False)

    print(f"Metrics saved to {metrics_csv_path}")

else:
    print("\nNo ground truth data found in the input CSV. Operating in Final Blind-Test Mode.")


--- Evaluation & Logging ---

Final answers saved directly into /content/drive/MyDrive/Agentic RAG Team 1/secret_test_set.csv
Trajectories logged to /content/drive/MyDrive/Agentic RAG Team 1/agent_trajectories.txt

FINAL SYSTEM METRICS:
 -> Exact Match (EM): 0.7333
 -> Retrieval Recall: 0.9333
 -> Average Steps-to-Answer: 1.8000
 -> Premature-Stops (No Context): 0.2500
 -> Reasoning Failures (Had Context): 0.7500
Metrics saved to /content/drive/MyDrive/Agentic RAG Team 1/evaluation_metrics.csv
